In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Core series data: deduped on natural key, basic quality expectations
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.silver.silver_bls_pr")
@dp.expect_or_drop("valid_year", "year BETWEEN 1900 AND 2100")
@dp.expect_or_drop("non_null_value", "value IS NOT NULL")
def silver_bls_pr():
    return (
        dp.read("rearc_bls.bronze.bronze_bls_pr")
        .dropDuplicates(["series_id", "year", "period"])
    )


# ---------------------------------------------------------------------------
# Series metadata + lookup tables: pass-through, trimmed already in Bronze
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.silver.silver_bls_series")
def silver_bls_series():
    return dp.read("rearc_bls.bronze.bronze_bls_series").dropDuplicates(["series_id"])


@dp.table(name="rearc_bls.silver.silver_bls_sector")
def silver_bls_sector():
    return dp.read("rearc_bls.bronze.bronze_bls_sector")


@dp.table(name="rearc_bls.silver.silver_bls_class")
def silver_bls_class():
    return dp.read("rearc_bls.bronze.bronze_bls_class")


@dp.table(name="rearc_bls.silver.silver_bls_measure")
def silver_bls_measure():
    return dp.read("rearc_bls.bronze.bronze_bls_measure")


@dp.table(name="rearc_bls.silver.silver_bls_duration")
def silver_bls_duration():
    return dp.read("rearc_bls.bronze.bronze_bls_duration")


# ---------------------------------------------------------------------------
# Population
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.silver.silver_population")
def silver_population():
    return (
        dp.read("rearc_bls.bronze.bronze_population")
        .select(
            F.col("Year").cast("int").alias("year"),
            F.col("Population").cast("long").alias("population"),
        )
        .dropDuplicates(["year"])
    )